# Project 3B - Water Pipe Failure
## Dataset Review / Initial Exploration

### Purpose:
This file is used for initial dataset loading and review.
The goal is to check whether each dataset is suitable for our project.

### Basic checks include:
- file loads successfully
- number of rows / columns
- column names / schema
- data types
- sample records
- missing values
- whether fields align with our project scope

### Datasets in review may include:
- Melbourne water main & soil / environmental data (Shival)
- Historical failure datasets
-- Netherlands & Kitchener (Anne)
-- Bozeman (Joe)

### Note:
This file is for early data exploration only.
It is not the final modelling script.


In [2]:
#Import all required packages
import pandas as pd
import numpy as np

### Melbourne Water Mains datasets (Shival)

TBD

### Melbourne Soil/Environmental datasets (Shival)

TBD

### Netherlands pipe failure datasets (Anne)

In [4]:
#Import Netherlands dataset

#Load pickle file
mains_nether = pd.read_pickle("data/Netherlands_watermains.pickle")
mains_nether = mains_nether.drop(columns='unit_ID')

#Load npy file
breaks_nether = np.load("data/Netherlands_watermainbreaks.npy")

#Check
print(mains_nether.head())
print(breaks_nether.shape)

           FUNCTIE_LE NETWERK MATERIAAL  UITWENDIGE  INWENDIGE_  NOMINALE_D  \
0  Distributieleiding      DN       PVC       110.0       103.4       110.0   
1  Distributieleiding      DN       PVC       110.0       103.4       110.0   
2  Distributieleiding      DN       PVC       110.0       103.4       110.0   
3  Distributieleiding      DN       PVC       110.0       103.4       110.0   
4  Distributieleiding      DN       PVC       110.0       104.6       110.0   

      Age  LENGTE_GIS  Aansluiting   distance       Soil_type  Vegetation  
0  1987.0  1050.73858     0.761369   8.185475            Zand       Rural  
1  1987.0   730.50514     1.095133   4.982359      Combinatie  High green  
2  1987.0   192.96527     5.700508   4.088136  Bebouwing, enz       Rural  
3  1987.0    59.62482     8.385770   7.384946  Bebouwing, enz       Rural  
4  1987.0   693.61399     0.576690  16.780591      Combinatie       Rural  
(10203, 6, 2)


In [8]:
#npy is a 3D array, not a 2D table
#So need to reshape it first

#Reshape from (10203, 6, 2) to (10203, 12)
breaks_nether = breaks_nether.reshape(breaks_nether.shape[0], -1)

#Convert Numpy array to Dataframe
breaks_nether_df = pd.DataFrame(breaks_nether)

#Check
print(breaks_nether_df.head())
print(breaks_nether_df.shape)
print(breaks_nether_df.info())

    0    1    2    3    4    5    6    7    8    9    10     11
0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  1.0  0.0  2.0  29.72
1  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  1.0  0.0  3.0  29.84
2  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  1.0  0.0  4.0  27.31
3  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  1.0  0.0  4.0  27.17
4  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  1.0  0.0  5.0  31.18
(10203, 12)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10203 entries, 0 to 10202
Data columns (total 12 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   0       10203 non-null  float64
 1   1       10203 non-null  float64
 2   2       10203 non-null  float64
 3   3       10203 non-null  float64
 4   4       10203 non-null  float64
 5   5       10203 non-null  float64
 6   6       10203 non-null  float64
 7   7       10203 non-null  float64
 8   8       10203 non-null  float64
 9   9       10203 non-null  float64
 10  10      10203 non-null  float64
 11  11      1

**Pros**

- Cleaner research dataset, already prepared for modelling
- Has both profile features + historical failure sequence
- Includes richer failure-history structure
- Better for time-to-next-failure analysis
- May already include some environment-related features (soil/vegetation/water hammer proxy)

**Cons**

- Built more for RNN / sequence model, not ideal for RF/XGBoost
- event_set.npy is harder to use in simple tabular ML
- Need extra work to convert sequence data into flat features
- Less natural for “high-risk segment ranking” compared to Kitchener

### Kitchener pipe failure datasets (Anne)

In [12]:
#Import Kitchener dataset
breaks_kit = pd.read_csv("data/Kitchener_watermainbreaks.csv")
mains_kit = pd.read_csv("data/Kitchener_watermains.csv")

#Quick check
print("Breaks shape:", breaks_kit.shape)
print("Mains shape:", mains_kit.shape)

print("\nBreaks columns:")
print(breaks_kit.columns)

print("\nMains columns:")
print(mains_kit.columns)

print("\nBreaks head:")
print(breaks_kit.head())

print("\nMains head:")
print(mains_kit.head())

Breaks shape: (2994, 37)
Mains shape: (16163, 27)

Breaks columns:
Index(['OBJECTID', 'Wat Break Incident ID', 'Incident date',
       'Type of Asset Broken', 'Does the road need to be closed?',
       'Does the sidewalk need to be closed?', 'Estimated Hours for Repair',
       'Estimated Number of Units Impacted', 'CW Service Request Number',
       'Current status of the break', 'Status last updated date',
       'CW Workorder #', 'Date operations was returned to normal service',
       'Nature of Break', 'Apparent cause of break', 'Repair Type',
       'Type of Planned Maintenance', 'List Valves Closed',
       'List Valves Opened', 'List Hydrants Called Out',
       'List Hydrants Called Back In', 'Categorization of the Break',
       'Road Segment ID', 'Closest Civic Number', 'Street', 'Related Asset ID',
       'Related Asset Depth (m)', 'Depth of Frost (m)', 'Asset Size (cm)',
       'Year Asset Installed', 'Asset Material', 'Asset Exists', 'GLOBALID',
       'UPDATE_BY', 'UPDAT

**Pros**

- Has both water main asset data + break history
- Easy to build asset-level risk model
- Easier to explain and turn into maintenance recommendations

**Cons**

- Soil/environment data not included in current files
- Need extra feature engineering for repair history
- Need GIS/environment data integration if we want soil or environmental factors

### Bozeman pipe failure datasets (Joe)

TBD

In [13]:
#Import Bozeman dataset
watermains = pd.read_csv("/Users/homedesk/Documents/S779/2026/SIT764/capstone/MOP-Code/Playground/Project_3B_T126/data/Bozeman_watermainbreaks.csv")
print(watermains.columns.tolist())


['X', 'Y', 'OBJECTID_1', 'Break_Date', 'Address', 'Pipe_Size', 'Type_of_Leak', 'Cost', 'WOID', 'FACID', 'PIPE_INSTALL_DA', 'Detected']


In [21]:

print(watermains.head(20).to_string())

                X             Y  OBJECTID_1              Break_Date                              Address  Pipe_Size                     Type_of_Leak     Cost      WOID   FACID  PIPE_INSTALL_DA Detected
0   496675.559596  5.058910e+06           1  2013/01/28 00:00:00+00                            431 N 4TH        6.0                        Corrosion  3273.33   86318.0   334.0           1948.0       No
1   496631.156500  5.059796e+06           2  2012/08/11 00:00:00+00                   1126 N 7TH (KMART)        8.0               Longitudinal Break   286.10   80400.0   204.0           1974.0       No
2   496352.228196  5.060142e+06           3  2002/02/15 00:00:00+00            5 Baxter Lane Holiday Inn        4.0             Pipe Failure at Bend   770.57       NaN   281.0           2001.0       No
3   496383.946396  5.058829e+06           4  1997/07/07 00:00:00+00            Seventh Av N/Villard St W        6.0  Rock in hyd nozzel - hammer N/C  2301.76       NaN   247.0           1963.0